# Shallow xLSTM Benchmark with Optuna & Symlog

In [ ]:
!pip install optuna xlstm

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna

from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    sLSTMBlockConfig,
    mLSTMBlockConfig,
    sLSTMLayerConfig,
    mLSTMLayerConfig
)


DATA_PATH = Path("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 70
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021
OPTUNA_TRIALS = 70
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Data Loading - Improved Pipeline

In [ ]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    try:
        df = pd.read_csv(data_path)
    except FileNotFoundError:
        df = pd.read_csv("Merged_Dataset_yoy.csv")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()

    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            historical_data = company_data[company_data["Date"] <= year_end_date]
            if len(historical_data) < lookback_days:
                continue

            window = historical_data.tail(lookback_days).copy()

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue
            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)
            data_records.append({
                "company": seq["company"], "year": seq["year"], "year_end_date": seq["year_end_date"],
                "target": target_name, "label_value": label_value,
                "window_data": seq["window_data"]
            })

    return pd.DataFrame(data_records), feature_cols

def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int):
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

# --- LOCK IN BEST PIPELINE: SYMLOG & ROBUST SCALER ---
improved_data_df = data_df.copy()
idx_value = improved_data_df["target"].isin(PREDICTION_TARGETS)
y_val = improved_data_df.loc[idx_value, "label_value"]
improved_data_df.loc[idx_value, "label_value"] = np.sign(y_val) * np.log1p(np.abs(y_val))

train_data_imp, val_data_imp, test_data_imp = split_by_year(improved_data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

selected_indices = [feature_cols.index(f) for f in feature_cols]
robust_scaler = RobustScaler()
train_windows_imp = np.vstack([row[:, selected_indices] for row in train_data_imp["window_data"]])
robust_scaler.fit(train_windows_imp)
print(f"Data Processed. Training sets: {len(train_data_imp)}")

Data Processed. Training sets: 3168


## Dataset and Multi-Task Preparation

In [ ]:
def extract_mtl_df(df: pd.DataFrame) -> pd.DataFrame:
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        window_data = group.iloc[0]["window_data"]
        year_end_date = group.iloc[0]["year_end_date"]
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": year_end_date,
            "window_data": window_data, "label_value": label_value
        })
    return pd.DataFrame(mtl_records)

mtl_train_data = extract_mtl_df(train_data_imp)
mtl_val_data = extract_mtl_df(val_data_imp)
mtl_test_data = extract_mtl_df(test_data_imp)

class ImprovedYoYDatasetMTL(Dataset):
    def __init__(self, data_df, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler: window = self.scaler.transform(window)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool), len(window)

def collate_fn_improved_mtl(batch):
    batch.sort(key=lambda x: x[3], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    masks = torch.stack([x[2] for x in batch])
    lengths = torch.tensor([x[3] for x in batch])
    # xLSTM does not use torch packing natively. We feed it zero-padded batches.
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, lengths

## xLSTM Model Architecture definition

Since `xLSTM` relies strictly on matrix processing, zero-padded `batch_first` tensors are used instead of `PyTorch pack_padded` utilities, taking care to extract only the actual final hidden state per element based on the masking lengths.

In [ ]:
class ImprovedShallow_xLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), dropout: float = 0.2, block_type: str = 'mlstm'):
        super().__init__()
        self.proj = nn.Linear(input_size, hidden_size)
        slstm_config = sLSTMLayerConfig(backend='vanilla')

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig()),
            context_length=400, # Large enough to encompass max lookback sequence limits
            num_blocks=1, # Depth 1, strictly comparing to shallow LSTM
            embedding_dim=hidden_size,
            slstm_at=[0] if block_type == 'slstm' else []
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        # x shape: [B, MaxSeq, input_size]
        B, S, _ = x.shape
        x_proj = self.proj(x)

        # Output strictly structured as [B, S, hidden_dim]
        out = self.xlstm(x_proj)

        # Dynamically recover the 'last valid hidden step' taking sequence padding into account
        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1) # [B, hidden_size]

        hidden = self.dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

In [ ]:
class TunableHybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int, mlstm_proj_factor: float,
                 mlstm_num_heads: int, slstm_hidden_size: int, dropout_mlstm: float,
                 dropout_slstm: float, dropout_global: float, num_targets: int = len(PREDICTION_TARGETS)):
        super().__init__()
        self.proj = nn.Linear(input_size, embedding_dim)

        slstm_config = sLSTMLayerConfig(
            backend='vanilla',
            hidden_size=slstm_hidden_size,
            dropout=dropout_slstm
        )

        mlstm_config = mLSTMLayerConfig(
            num_heads=mlstm_num_heads,
            proj_factor=mlstm_proj_factor,
            dropout=dropout_mlstm
        )

        # ... (rest of the __init__ and forward pass remain exactly the same)

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mlstm_config),
            context_length=400,
            num_blocks=3,
            embedding_dim=embedding_dim,
            slstm_at=[2] # Block 0 & 1 are mLSTM, Block 2 is sLSTM
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.head_dropout = nn.Dropout(p=dropout_global)
        self.head = nn.Linear(embedding_dim, num_targets)

    def forward(self, x, lengths):
        B, S, _ = x.shape
        out = self.proj(x)
        out = self.xlstm(out)

        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1)

        hidden = self.head_dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

def objective_hybrid(trial):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
    # Core transport bus dimension
    embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

    # mLSTM specific settings (Matrix capacity)
    mlstm_proj_factor = trial.suggest_categorical("mlstm_proj_factor", [1.0, 1.5, 2.0])
    mlstm_num_heads = trial.suggest_categorical("mlstm_num_heads", [2, 4])

    # Prune immediately if embedding_dim cannot be evenly divided by heads
    if embedding_dim % mlstm_num_heads != 0:
        raise optuna.TrialPruned()

    # sLSTM specific settings (Scalar tracking capacity)
    slstm_hidden_size = trial.suggest_categorical("slstm_hidden_size", [16, 32, 64])

    # Regularization
    dropout_mlstm = trial.suggest_float("dropout_mlstm", 0.1, 0.4)
    dropout_slstm = trial.suggest_float("dropout_slstm", 0.1, 0.4)
    dropout_global = trial.suggest_float("dropout_global", 0.2, 0.5)

    # Optimizer & Training
    lr_mlstm = trial.suggest_float("lr_mlstm", 1e-4, 5e-3, log=True)
    lr_slstm = trial.suggest_float("lr_slstm", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    train_ds = ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices)
    val_ds = ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

    model = TunableHybrid_xLSTM(
        input_size=len(selected_indices),
        embedding_dim=embedding_dim,
        mlstm_proj_factor=mlstm_proj_factor,
        mlstm_num_heads=mlstm_num_heads,
        slstm_hidden_size=slstm_hidden_size,
        dropout_mlstm=dropout_mlstm,
        dropout_slstm=dropout_slstm,
        dropout_global=dropout_global
    )

    mlstm_params = [p for n, p in model.named_parameters() if 'mlstm' in n]
    slstm_params = [p for n, p in model.named_parameters() if 'slstm' in n]
    other_params = [p for n, p in model.named_parameters() if 'mlstm' not in n and 'slstm' not in n]

    optimizer = torch.optim.AdamW([
        {'params': mlstm_params, 'lr': lr_mlstm},
        {'params': slstm_params, 'lr': lr_slstm},
        {'params': other_params, 'lr': lr_mlstm}
    ], weight_decay=weight_decay)

    criterion = RealScaleL1Loss()
    val_loss = train_model_v2(model, train_loader, val_loader, criterion, optimizer,
                           epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE, trial=trial)

    return val_loss

## Optuna & Training Loop Helper

In [ ]:
def inverse_symlog_tensor(values):
    return torch.sign(values) * torch.expm1(torch.abs(values))

class RealScaleL1Loss(nn.Module):
    def forward(self, preds_symlog, targets_symlog):
        preds_real = inverse_symlog_tensor(preds_symlog)
        targets_real = inverse_symlog_tensor(targets_symlog)
        return torch.mean(torch.abs(preds_real - targets_real))

def train_model(model, train_loader, val_loader, criterion, lr, weight_decay, epochs, patience, device, trial=None):
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5
    )

    best_loss = float('inf')
    best_epoch = -1
    early_stop_counter = 0
    history = {
        "train_loss": [],
        "val_loss": [],
        "lr": []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_valid_count = 0

        for x_batch, y_batch, mask_batch, lengths in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)
            lengths = lengths.to(device)

            if mask_batch.sum() == 0:
                continue

            optimizer.zero_grad()
            preds = model(x_batch, lengths)
            valid_count = mask_batch.sum().item()

            loss = criterion(
                preds[mask_batch],
                y_batch[mask_batch]
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            train_loss += loss.item() * valid_count
            train_valid_count += valid_count

        train_loss /= max(1, train_valid_count)

        model.eval()

        val_loss = 0.0
        val_valid_count = 0

        with torch.no_grad():

            for x_batch, y_batch, mask_batch, lengths in val_loader:

                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                mask_batch = mask_batch.to(device)
                lengths = lengths.to(device)

                if mask_batch.sum() == 0:
                    continue

                preds = model(x_batch, lengths)

                valid_count = mask_batch.sum().item()

                loss = criterion(
                    preds[mask_batch],
                    y_batch[mask_batch]
                )

                val_loss += loss.item() * valid_count
                val_valid_count += valid_count

        val_loss /= max(1, val_valid_count)

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        if trial is not None:
            trial.report(val_loss, epoch)

            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_loss < best_loss:

            best_loss = val_loss
            best_epoch = epoch

            early_stop_counter = 0

            torch.save(
                model.state_dict(),
                "best_xlstm_model.pth"
            )

        else:

            early_stop_counter += 1

            if early_stop_counter >= patience:
                break

    return {
        "best_val_loss": best_loss,
        "best_epoch": best_epoch,
        "history": history
    }

def train_model_v2(model, train_loader, val_loader, criterion, optimizer, epochs, patience, device, trial=None):
    model.to(device)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_loss = float('inf')
    early_stop_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_valid_count = 0
        for x_batch, y_batch, mask_batch, lengths in train_loader:
            x_batch, y_batch, mask_batch, lengths = x_batch.to(device), y_batch.to(device), mask_batch.to(device), lengths.to(device)

            if mask_batch.sum() == 0:
                continue

            optimizer.zero_grad()
            preds = model(x_batch, lengths)
            valid_count = mask_batch.sum().item()
            loss = criterion(preds[mask_batch], y_batch[mask_batch])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * valid_count
            train_valid_count += valid_count

        train_loss /= max(1, train_valid_count)

        model.eval()
        val_loss = 0.0
        val_valid_count = 0
        with torch.no_grad():
            for x_batch, y_batch, mask_batch, lengths in val_loader:
                x_batch, y_batch, mask_batch, lengths = x_batch.to(device), y_batch.to(device), mask_batch.to(device), lengths.to(device)

                if mask_batch.sum() == 0:
                    continue

                preds = model(x_batch, lengths)
                valid_count = mask_batch.sum().item()
                loss = criterion(preds[mask_batch], y_batch[mask_batch])
                val_loss += loss.item() * valid_count
                val_valid_count += valid_count

        val_loss /= max(1, val_valid_count)
        scheduler.step(val_loss)

        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_loss < best_loss:
            best_loss = val_loss
            early_stop_counter = 0
            torch.save(model.state_dict(), 'best_xlstm_model.pth')
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                break

    return best_loss

def evaluate_real_metrics(model, test_loader, device):
    model.eval()
    total_absolute_error = 0.0
    total_squared_error = 0.0
    total_samples = 0

    with torch.no_grad():
        for x_batch, y_batch_symlog, mask_batch, lengths in test_loader:
            x_batch, y_batch_symlog, mask_batch, lengths = x_batch.to(device), y_batch_symlog.to(device), mask_batch.to(device), lengths.to(device)
            if mask_batch.sum() == 0: continue

            preds_symlog = model(x_batch, lengths)
            valid_preds_symlog = preds_symlog[mask_batch]
            valid_y_symlog = y_batch_symlog[mask_batch]

            # Inverse Symlog transform back to the scale of human values
            preds_real = torch.sign(valid_preds_symlog) * torch.expm1(torch.abs(valid_preds_symlog))
            y_real = torch.sign(valid_y_symlog) * torch.expm1(torch.abs(valid_y_symlog))

            absolute_error = torch.abs(preds_real - y_real).sum().item()
            squared_error = torch.pow(preds_real - y_real, 2).sum().item()

            total_absolute_error += absolute_error
            total_squared_error += squared_error
            total_samples += valid_y_symlog.size(0)

    real_mae = total_absolute_error / total_samples
    real_rmse = (total_squared_error / total_samples) ** 0.5
    print(f"Test MAE  (Real Scale): {real_mae:.4f}")
    print(f"Test RMSE (Real Scale): {real_rmse:.4f}")
    print("="*50 + "\n")

    return real_mae, real_rmse

def objective(trial):
    # Hyperparams natively focused on xLSTM properties and Regularization
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # xLSTM uniquely offers distinct recurrent engines (matrix vs scalar variants)
    block_type = trial.suggest_categorical('block_type', ['mlstm', 'slstm'])

    train_ds = ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices)
    val_ds = ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

    model = ImprovedShallow_xLSTM(
        input_size=len(selected_indices),
        hidden_size=hidden_size,
        num_targets=len(PREDICTION_TARGETS),
        dropout=dropout,
        block_type=block_type
    )

    # Locked onto our winner criteria from LSTM Tests:
    criterion = nn.L1Loss()

    val_loss = train_model(model, train_loader, val_loader, criterion, lr, weight_decay,
                           epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE, trial=trial)

    return val_loss

def evaluate_per_target_metrics(model, test_loader, device, target_names):
    model.eval()
    target_mae = {name: 0.0 for name in target_names}
    target_samples = {name: 0 for name in target_names}

    with torch.no_grad():
        for x_batch, y_batch_symlog, mask_batch, lengths in test_loader:
            x_batch, y_batch_symlog, mask_batch, lengths = x_batch.to(device), y_batch_symlog.to(device), mask_batch.to(device), lengths.to(device)

            preds_symlog = model(x_batch, lengths)

            for i, name in enumerate(target_names):
                valid_mask = mask_batch[:, i]
                if valid_mask.sum() == 0: continue

                y_true_sym = y_batch_symlog[valid_mask, i]
                y_pred_sym = preds_symlog[valid_mask, i]

                y_true_real = torch.sign(y_true_sym) * torch.expm1(torch.abs(y_true_sym))
                y_pred_real = torch.sign(y_pred_sym) * torch.expm1(torch.abs(y_pred_sym))

                mae = torch.abs(y_true_real - y_pred_real).sum().item()

                target_mae[name] += mae
                target_samples[name] += valid_mask.sum().item()

    print("--- MAE PER TARGET (REAL SCALE) ---")
    results = {}
    for name in target_names:
        final_mae = target_mae[name] / target_samples[name] if target_samples[name] > 0 else float('nan')
        results[name] = final_mae
        print(f"{name:12} MAE: {final_mae:.4f}")
    print("="*35)
    return results


# Models

In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective, n_trials=OPTUNA_TRIALS)

[I 2026-06-01 09:26:29,125] A new study created in memory with name: no-name-df765e31-c500-4108-b710-f7abe4e48972
[I 2026-06-01 09:28:01,928] Trial 0 finished with value: 0.6845970360438028 and parameters: {'hidden_size': 128, 'dropout': 0.21405395398367744, 'lr': 0.0008014092608314635, 'weight_decay': 0.003751164975443517, 'batch_size': 64, 'block_type': 'mlstm'}. Best is trial 0 with value: 0.6845970360438028.
[I 2026-06-01 09:28:27,299] Trial 1 finished with value: 0.6548038260142008 and parameters: {'hidden_size': 64, 'dropout': 0.2877650565898008, 'lr': 0.002972129991664589, 'weight_decay': 0.001342573317072393, 'batch_size': 128, 'block_type': 'mlstm'}. Best is trial 1 with value: 0.6548038260142008.
[I 2026-06-01 09:30:38,223] Trial 2 finished with value: 0.608865385055542 and parameters: {'hidden_size': 128, 'dropout': 0.4389044294092056, 'lr': 0.00023201701988688928, 'weight_decay': 0.0007534307319340079, 'batch_size': 64, 'block_type': 'mlstm'}. Best is trial 2 with value: 0.

In [ ]:

print("OPTUNA BEST INITIALIZATION PARAMETERS")
print(f"Best Trial Validation Loss (Symlog Space): {study.best_trial.value:.4f}")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*50 + "\n")

optimal_batch_size = study.best_trial.params["batch_size"]
test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=optimal_batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

final_model = ImprovedShallow_xLSTM(
    input_size=len(selected_indices),
    hidden_size=study.best_trial.params["hidden_size"],
    num_targets=len(PREDICTION_TARGETS),
    dropout=study.best_trial.params["dropout"],
    block_type=study.best_trial.params["block_type"]
)

print("Executing final training pass to lock-in optimal weights...")
train_loader_final = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=optimal_batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_final = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=optimal_batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

_ = train_model(
    model=final_model,
    train_loader=train_loader_final,
    val_loader=val_loader_final,
    criterion=nn.L1Loss(),
    lr=study.best_trial.params["lr"],
    weight_decay=study.best_trial.params["weight_decay"],
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

final_model.load_state_dict(torch.torch.load('best_xlstm_model.pth', map_location=DEVICE))
real_mae, real_rmse = evaluate_real_metrics(final_model, test_loader, DEVICE)

OPTUNA BEST INITIALIZATION PARAMETERS
Best Trial Validation Loss (Symlog Space): 0.6073
  hidden_size: 32
  dropout: 0.3338511063722587
  lr: 0.000239105723304576
  weight_decay: 0.0006335917876427258
  batch_size: 32
  block_type: slstm

Executing final training pass to lock-in optimal weights...
Test MAE  (Real Scale): 1.5803
Test RMSE (Real Scale): 8.6107



In [ ]:
per_target_results = evaluate_per_target_metrics(final_model, test_loader, DEVICE, PREDICTION_TARGETS)

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.5274
Net_Income   MAE: 1.6016
ROA          MAE: 1.6116


In [ ]:
class Hybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), dropout: float = 0.2):
        super().__init__()
        self.proj = nn.Linear(input_size, hidden_size)
        slstm_config = sLSTMLayerConfig(backend='vanilla')

        # We configure 2 blocks: 1 mLSTM layer followed by 1 sLSTM layer
        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig()),
            context_length=400,
            num_blocks=2,
            embedding_dim=hidden_size,
            slstm_at=[1]  # Index 1 is the second block. Index 0 defaults to mLSTM.
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        # x shape: [B, MaxSeq, input_size]
        B, S, _ = x.shape
        x_proj = self.proj(x)

        # Output strictly structured as [B, S, hidden_dim]
        out = self.xlstm(x_proj)

        # Dynamically recover the 'last valid hidden step' taking sequence padding into account
        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1) # [B, hidden_size]

        hidden = self.dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

# Hybrid Parameters (Blending aspects from both best runs)
hybrid_params = {
    'hidden_size': 64,
    'dropout': 0.33,
    'lr': 0.001,
    'weight_decay': 0.002,
    'batch_size': 64
}

print("Training Hybrid xLSTM (mLSTM -> sLSTM)...")

test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=hybrid_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)
train_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=hybrid_params["batch_size"], shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=hybrid_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)

hybrid_model = Hybrid_xLSTM(
    input_size=len(selected_indices),
    hidden_size=hybrid_params["hidden_size"],
    num_targets=len(PREDICTION_TARGETS),
    dropout=hybrid_params["dropout"]
)

_ = train_model(
    model=hybrid_model,
    train_loader=train_loader_hybrid,
    val_loader=val_loader_hybrid,
    criterion=nn.L1Loss(),
    lr=hybrid_params["lr"],
    weight_decay=hybrid_params["weight_decay"],
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

hybrid_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))
print("\n--- HYBRID MODEL OVERALL RESULTS ---")
real_mae_hybrid, real_rmse_hybrid = evaluate_real_metrics(hybrid_model, test_loader, DEVICE)
per_target_results_hybrid = evaluate_per_target_metrics(hybrid_model, test_loader, DEVICE, PREDICTION_TARGETS)

Training Hybrid xLSTM (mLSTM -> sLSTM)...

--- HYBRID MODEL OVERALL RESULTS ---
Test MAE  (Real Scale): 1.5728
Test RMSE (Real Scale): 8.6135

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.5255
Net_Income   MAE: 1.5989
ROA          MAE: 1.5934


In [ ]:
class Hybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), dropout: float = 0.2):
        super().__init__()
        self.proj = nn.Linear(input_size, hidden_size)
        slstm_config = sLSTMLayerConfig(backend='vanilla')

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig()),
            context_length=400,
            num_blocks=3,
            embedding_dim=hidden_size,
            slstm_at=[2]
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        # x shape: [B, MaxSeq, input_size]
        B, S, _ = x.shape
        x_proj = self.proj(x)

        # Output strictly structured as [B, S, hidden_dim]
        out = self.xlstm(x_proj)

        # Dynamically recover the 'last valid hidden step' taking sequence padding into account
        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1) # [B, hidden_size]

        hidden = self.dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

hybrid_params = {
    'hidden_size': 64,
    'dropout': 0.33,
    'lr': 0.001,
    'weight_decay': 0.002,
    'batch_size': 64
}

print("Training Hybrid xLSTM (mLSTM -> mLSTM -> sLSTM)")

test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=hybrid_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)
train_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=hybrid_params["batch_size"], shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=hybrid_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)

hybrid_model = Hybrid_xLSTM(
    input_size=len(selected_indices),
    hidden_size=hybrid_params["hidden_size"],
    num_targets=len(PREDICTION_TARGETS),
    dropout=hybrid_params["dropout"]
)

total_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
print(f"Final number of parameters: {total_params:,}")

_ = train_model(
    model=hybrid_model,
    train_loader=train_loader_hybrid,
    val_loader=val_loader_hybrid,
    criterion=nn.L1Loss(),
    lr=hybrid_params["lr"],
    weight_decay=hybrid_params["weight_decay"],
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

hybrid_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))
print("\n--- HYBRID MODEL OVERALL RESULTS ---")
real_mae_hybrid, real_rmse_hybrid = evaluate_real_metrics(hybrid_model, test_loader, DEVICE)
per_target_results_hybrid = evaluate_per_target_metrics(hybrid_model, test_loader, DEVICE, PREDICTION_TARGETS)

Training Hybrid xLSTM (mLSTM -> mLSTM -> sLSTM)
Final number of parameters: 95,763

--- HYBRID MODEL OVERALL RESULTS ---
Test MAE  (Real Scale): 1.5723
Test RMSE (Real Scale): 8.6111

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.5228
Net_Income   MAE: 1.6056
ROA          MAE: 1.5879


In [ ]:
class ImprovedHybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int = 64, num_targets: int = len(PREDICTION_TARGETS)):
        super().__init__()
        self.proj = nn.Linear(input_size, embedding_dim)

        # 1) sLSTM config (Strictly Trial 5)
        # Inside the block config we can explicitly override the internal hidden dimension
        # setting it to 32 exactly as Trial 5 suggested, and applying block-level dropout
        slstm_config = sLSTMLayerConfig(
            backend='vanilla',
            hidden_size=32,
            dropout=0.334    # Extracted from Trial 5
        )

        # 2) mLSTM config (Strictly Trial 63)
        # The xLSTM stack requires a shared "embedding_dim" connecting the layers (we use 64).
        # We can reach the desired 128 hidden_size from Trial 63 inside the mLSTM block
        # by using a proj_factor of 2.0 (embedding_dim 64 * 2.0 = 128 inner dimension)
        mlstm_config = mLSTMLayerConfig(
            num_heads=4,
            proj_factor=2.0, # Scales internal capacity to 128
            dropout=0.330    # Extracted from Trial 63
        )

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(
                slstm=slstm_config,
            ),
            mlstm_block=mLSTMBlockConfig(
                mlstm=mlstm_config,
            ),
            context_length=400,
            num_blocks=3,
            embedding_dim=embedding_dim, # 64 -> Transport bus between blocks
            slstm_at=[2]
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.head_dropout = nn.Dropout(p=0.33)
        self.head = nn.Linear(embedding_dim, num_targets)

    def forward(self, x, lengths):
        B, S, _ = x.shape
        out = self.proj(x)
        out = self.xlstm(out)

        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1)

        hidden = self.head_dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

# Because the optimizer settings (lr, weight_decay, batch_size) apply to the whole loop,
# we still need to pick a global setup here. We will use the aggressive ones from mLSTM (Trial 63)
# since it occupies 2 of the 3 blocks, but you could tune this later.
lr_global = 0.0044
weight_decay_global = 0.0054
batch_size_global = 64

print("Training Improved Hybrid xLSTM (Explicitly Split Configurations)")

test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)
train_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)

hybrid_model = ImprovedHybrid_xLSTM(
    input_size=len(selected_indices),
    embedding_dim=64,
    num_targets=len(PREDICTION_TARGETS)
)

total_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
print(f"Final number of parameters: {total_params:,}")

_ = train_model(
    model=hybrid_model,
    train_loader=train_loader_hybrid,
    val_loader=val_loader_hybrid,
    criterion=nn.L1Loss(),
    lr=lr_global,
    weight_decay=weight_decay_global,
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

hybrid_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))
print("\n--- IMPROVED HYBRID MODEL OVERALL RESULTS ---")
real_mae_hybrid, real_rmse_hybrid = evaluate_real_metrics(hybrid_model, test_loader, DEVICE)
per_target_results_hybrid = evaluate_per_target_metrics(hybrid_model, test_loader, DEVICE, PREDICTION_TARGETS)

Training Improved Hybrid xLSTM (Explicitly Split Configurations)
Final number of parameters: 95,763

--- IMPROVED HYBRID MODEL OVERALL RESULTS ---
Test MAE  (Real Scale): 1.5554
Test RMSE (Real Scale): 8.6138

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4919
Net_Income   MAE: 1.5843
ROA          MAE: 1.5895


In [ ]:
print("Starting explicit Hybrid xLSTM Optimization Sweep...")

hybrid_study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
hybrid_study.optimize(objective_hybrid, n_trials=OPTUNA_TRIALS)

print("\n" + "="*50)
print("OPTUNA BEST HYBRID INITIALIZATION PARAMETERS")
print(f"Best Trial Validation MAE (Real Scale): {hybrid_study.best_trial.value:.4f}")
for key, value in hybrid_study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*50 + "\n")

[I 2026-06-05 13:41:50,183] A new study created in memory with name: no-name-f8d0ec9d-abc8-43e0-8680-706a9010fc42


Starting explicit Hybrid xLSTM Optimization Sweep...


[I 2026-06-05 13:47:44,138] Trial 0 finished with value: 7.874449814821011 and parameters: {'embedding_dim': 64, 'mlstm_proj_factor': 1.0, 'mlstm_num_heads': 4, 'slstm_hidden_size': 16, 'dropout_mlstm': 0.3067837827314225, 'dropout_slstm': 0.3815337183283144, 'dropout_global': 0.31446239462450276, 'lr_mlstm': 0.0009044836851796109, 'lr_slstm': 0.0011126522092438213, 'weight_decay': 0.00095779464928545, 'batch_size': 128}. Best is trial 0 with value: 7.874449814821011.
[I 2026-06-05 14:00:02,444] Trial 1 finished with value: 7.85955544428489 and parameters: {'embedding_dim': 32, 'mlstm_proj_factor': 1.0, 'mlstm_num_heads': 2, 'slstm_hidden_size': 32, 'dropout_mlstm': 0.3479793511884824, 'dropout_slstm': 0.19569669243013954, 'dropout_global': 0.4252054969447627, 'lr_mlstm': 0.00028122268083458654, 'lr_slstm': 0.0018683275533699138, 'weight_decay': 2.8412934472274956e-05, 'batch_size': 32}. Best is trial 1 with value: 7.85955544428489.
[I 2026-06-05 14:05:17,930] Trial 2 finished with val

[I 2026-06-05 15:22:42,017] Trial 21 finished with value: 7.773644468952534 and parameters: {'embedding_dim': 64, 'mlstm_proj_factor': 2.0, 'mlstm_num_heads': 2, 'slstm_hidden_size': 16, 'dropout_mlstm': 0.15708237336301523, 'dropout_slstm': 0.2893106032804826, 'dropout_global': 0.20028242114495576, 'lr_mlstm': 0.0014450682022411313, 'lr_slstm': 0.0018244409613728076, 'weight_decay': 4.220344465283051e-05, 'batch_size': 64}. Best is trial 21 with value: 7.773644468952534.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# 2. Optimal parameters extracted from Optuna Trial 21
best_params = {
    'embedding_dim': 64,
    'mlstm_proj_factor': 2.0,
    'mlstm_num_heads': 2,
    'slstm_hidden_size': 16,
    'dropout_mlstm': 0.15708237336301523,
    'dropout_slstm': 0.2893106032804826,
    'dropout_global': 0.20028242114495576,
    'lr_mlstm': 0.0014450682022411313,
    'lr_slstm': 0.0018244409613728076,
    'weight_decay': 4.220344465283051e-05,
    'batch_size': 64
}

# Increase patience and epochs to allow the decoupled learning rates to converge fully
FINAL_EPOCHS = 150
FINAL_PATIENCE = 20

print("Preparing DataLoaders for Final Training...")

# 3. Instantiate DataLoaders
train_ds = ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices)
val_ds = ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices)
test_ds = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)

train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], shuffle=False, collate_fn=collate_fn_improved_mtl)
test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'], shuffle=False, collate_fn=collate_fn_improved_mtl)

print("Initializing Final Hybrid xLSTM Model...")

# 4. Build the Final Model architecture
final_model = TunableHybrid_xLSTM(
    input_size=len(selected_indices),
    embedding_dim=best_params['embedding_dim'],
    mlstm_proj_factor=best_params['mlstm_proj_factor'],
    mlstm_num_heads=best_params['mlstm_num_heads'],
    slstm_hidden_size=best_params['slstm_hidden_size'],
    dropout_mlstm=best_params['dropout_mlstm'],
    dropout_slstm=best_params['dropout_slstm'],
    dropout_global=best_params['dropout_global']
).to(DEVICE)

# 5. Create the Decoupled Optimizer to feed different engines with different rates
mlstm_params = [p for n, p in final_model.named_parameters() if 'mlstm' in n]
slstm_params = [p for n, p in final_model.named_parameters() if 'slstm' in n]
other_params = [p for n, p in final_model.named_parameters() if 'mlstm' not in n and 'slstm' not in n]

final_optimizer = torch.optim.AdamW([
    {'params': mlstm_params, 'lr': best_params['lr_mlstm']},
    {'params': slstm_params, 'lr': best_params['lr_slstm']},
    {'params': other_params, 'lr': best_params['lr_mlstm']} # Default projections/head to the mLSTM rate
], weight_decay=best_params['weight_decay'])

criterion = RealScaleL1Loss()

print("Starting Final Training Run (Trial 27 Parameters)...")

# 6. Train the Model (trial=None ensures it runs as a standard training loop)
final_val_loss = train_model_v2(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=final_optimizer,
    epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    device=DEVICE,
    trial=None
)

print(f"\nFinal Validation Loss achieved: {final_val_loss:.4f}")

# 7. Evaluate on Unseen Test Set
print("\n" + "="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)

# Load the mathematically optimal weights saved by early stopping
final_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))

real_mae, real_rmse = evaluate_real_metrics(final_model, test_loader, DEVICE)
per_target_results = evaluate_per_target_metrics(final_model, test_loader, DEVICE, PREDICTION_TARGETS)

Preparing DataLoaders for Final Training...
Initializing Final Hybrid xLSTM Model...
Starting Final Training Run (Trial 27 Parameters)...

Final Validation Loss achieved: 7.7736

FINAL TEST SET EVALUATION
Test MAE  (Real Scale): 1.6020
Test RMSE (Real Scale): 8.6186

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.5165
Net_Income   MAE: 1.6389
ROA          MAE: 1.6498


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# 2. Optimal parameters extracted from Optuna Trial 27
best_params = {
    'embedding_dim': 64,
    'mlstm_proj_factor': 2,
    'mlstm_num_heads': 2,
    'slstm_hidden_size': 16,
    'dropout_mlstm': 0.16339457976146657,
    'dropout_slstm': 0.2775634291780588,
    'dropout_global': 0.49860109436399425,
    'lr_mlstm': 0.00016151041276996845,
    'lr_slstm': 0.0003188669665410082,
    'weight_decay': 0.00015047576244664628,
    'batch_size': 32
}

# Increase patience and epochs to allow the decoupled learning rates to converge fully
FINAL_EPOCHS = 150
FINAL_PATIENCE = 20

print("Preparing DataLoaders for Final Training...")

# 3. Instantiate DataLoaders
train_ds = ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices)
val_ds = ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices)
test_ds = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)

train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], shuffle=False, collate_fn=collate_fn_improved_mtl)
test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'], shuffle=False, collate_fn=collate_fn_improved_mtl)

print("Initializing Final Hybrid xLSTM Model...")

# 4. Build the Final Model architecture
final_model = TunableHybrid_xLSTM(
    input_size=len(selected_indices),
    embedding_dim=best_params['embedding_dim'],
    mlstm_proj_factor=best_params['mlstm_proj_factor'],
    mlstm_num_heads=best_params['mlstm_num_heads'],
    slstm_hidden_size=best_params['slstm_hidden_size'],
    dropout_mlstm=best_params['dropout_mlstm'],
    dropout_slstm=best_params['dropout_slstm'],
    dropout_global=best_params['dropout_global']
).to(DEVICE)

# 5. Create the Decoupled Optimizer to feed different engines with different rates
mlstm_params = [p for n, p in final_model.named_parameters() if 'mlstm' in n]
slstm_params = [p for n, p in final_model.named_parameters() if 'slstm' in n]
other_params = [p for n, p in final_model.named_parameters() if 'mlstm' not in n and 'slstm' not in n]

final_optimizer = torch.optim.AdamW([
    {'params': mlstm_params, 'lr': best_params['lr_mlstm']},
    {'params': slstm_params, 'lr': best_params['lr_slstm']},
    {'params': other_params, 'lr': best_params['lr_mlstm']} # Default projections/head to the mLSTM rate
], weight_decay=best_params['weight_decay'])

criterion = RealScaleL1Loss()

print("Starting Final Training Run (Trial 27 Parameters)...")

# 6. Train the Model (trial=None ensures it runs as a standard training loop)
final_val_loss = train_model_v2(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=final_optimizer,
    epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    device=DEVICE,
    trial=None
)

print(f"\nFinal Validation Loss achieved: {final_val_loss:.4f}")

# 7. Evaluate on Unseen Test Set
print("\n" + "="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)

# Load the mathematically optimal weights saved by early stopping
final_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))

real_mae, real_rmse = evaluate_real_metrics(final_model, test_loader, DEVICE)
per_target_results = evaluate_per_target_metrics(final_model, test_loader, DEVICE, PREDICTION_TARGETS)

Preparing DataLoaders for Final Training...
Initializing Final Hybrid xLSTM Model...
Starting Final Training Run (Trial 27 Parameters)...

Final Validation Loss achieved: 7.7865

FINAL TEST SET EVALUATION
Test MAE  (Real Scale): 1.6164
Test RMSE (Real Scale): 8.6248

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.6661
Net_Income   MAE: 1.5925
ROA          MAE: 1.5910


# Best model

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class ImprovedHybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int = 64, num_targets: int = len(PREDICTION_TARGETS)):
        super().__init__()
        self.proj = nn.Linear(input_size, embedding_dim)

        # 1) sLSTM config (Strictly Trial 5)
        # Inside the block config we can explicitly override the internal hidden dimension
        # setting it to 32 exactly as Trial 5 suggested, and applying block-level dropout
        slstm_config = sLSTMLayerConfig(
            backend='vanilla',
            hidden_size=32,
            dropout=0.334    # Extracted from Trial 5
        )

        # 2) mLSTM config (Strictly Trial 63)
        # The xLSTM stack requires a shared "embedding_dim" connecting the layers (we use 64).
        # We can reach the desired 128 hidden_size from Trial 63 inside the mLSTM block
        # by using a proj_factor of 2.0 (embedding_dim 64 * 2.0 = 128 inner dimension)
        mlstm_config = mLSTMLayerConfig(
            num_heads=4,
            proj_factor=2.0, # Scales internal capacity to 128
            dropout=0.330    # Extracted from Trial 63
        )

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(
                slstm=slstm_config,
            ),
            mlstm_block=mLSTMBlockConfig(
                mlstm=mlstm_config,
            ),
            context_length=400,
            num_blocks=3,
            embedding_dim=embedding_dim, # 64 -> Transport bus between blocks
            slstm_at=[2]
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.head_dropout = nn.Dropout(p=0.33)
        self.head = nn.Linear(embedding_dim, num_targets)

    def forward(self, x, lengths):
        B, S, _ = x.shape
        out = self.proj(x)
        out = self.xlstm(out)

        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1)

        hidden = self.head_dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

lr_global = 0.0044
weight_decay_global = 0.0054
batch_size_global = 64

print("Training Improved Hybrid xLSTM (Explicitly Split Configurations)")

test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)
train_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)

seeds = [42, 123, 456, 789, 1337]

maes = []
rmses = []
results = []

for seed in seeds:

    print(f"\n========== SEED {seed} ==========")

    seed_everything(seed)

    hybrid_model = ImprovedHybrid_xLSTM(
        input_size=len(selected_indices),
        embedding_dim=64,
        num_targets=len(PREDICTION_TARGETS)
    ).to(DEVICE)

    train_info = train_model(
        model=hybrid_model,
        train_loader=train_loader_hybrid,
        val_loader=val_loader_hybrid,
        criterion=RealScaleL1Loss(),
        lr=lr_global,
        weight_decay=weight_decay_global,
        epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=DEVICE
    )

    hybrid_model.load_state_dict(
        torch.load('best_xlstm_model.pth', map_location=DEVICE)
    )

    torch.save(hybrid_model.state_dict(), f"best_xlstm_seed_{seed}.pth")

    mae, rmse = evaluate_real_metrics(
        hybrid_model,
        test_loader,
        DEVICE
    )

    maes.append(mae)
    rmses.append(rmse)
    results.append({
        'seed': seed,
        'best_epoch': train_info['best_epoch'],
        'best_val_loss': train_info['best_val_loss'],
        'mae': mae,
        'rmse': rmse
    })

    print(
        f"Seed {seed} | "
        f"Epoch={train_info['best_epoch']} | "
        f"Val={train_info['best_val_loss']:.4f} | "
        f"MAE={mae:.4f}"
    )





print("\n==============================")
print("FINAL STATISTICS")
print("==============================")

print(f"MAE Mean : {np.mean(maes):.4f}")
print(f"MAE Std  : {np.std(maes, ddof=1):.4f}")

print(f"RMSE Mean: {np.mean(rmses):.4f}")
print(f"RMSE Std : {np.std(rmses, ddof=1):.4f}")


df = pd.DataFrame(results)

print("\n==============================")
print("DETAILED RESULTS")
print("==============================")

print(df.sort_values("mae"))

print("\nCorrelation:")
print(df[["best_val_loss", "mae"]].corr())

Training Improved Hybrid xLSTM (Explicitly Split Configurations)

========== SEED 42 ==========
Test MAE  (Real Scale): 1.5536
Test RMSE (Real Scale): 8.6101

Seed 42 | Epoch=19 | Val=7.8290 | MAE=1.5536

========== SEED 123 ==========
Test MAE  (Real Scale): 1.5563
Test RMSE (Real Scale): 8.6098

Seed 123 | Epoch=5 | Val=7.8314 | MAE=1.5563

========== SEED 456 ==========
Test MAE  (Real Scale): 1.6373
Test RMSE (Real Scale): 8.6183

Seed 456 | Epoch=30 | Val=7.8152 | MAE=1.6373

========== SEED 789 ==========
Test MAE  (Real Scale): 1.5628
Test RMSE (Real Scale): 8.6161

Seed 789 | Epoch=13 | Val=7.8181 | MAE=1.5628

========== SEED 1337 ==========
Test MAE  (Real Scale): 1.5460
Test RMSE (Real Scale): 8.6086

Seed 1337 | Epoch=7 | Val=7.8104 | MAE=1.5460

FINAL STATISTICS
MAE Mean : 1.5712
MAE Std  : 0.0375
RMSE Mean: 8.6126
RMSE Std : 0.0043

DETAILED RESULTS
   seed  best_epoch  best_val_loss       mae      rmse
4  1337           7       7.810427  1.545980  8.608628
0    42       

In [ ]:
class Ensemble_xLSTM(nn.Module):
    def __init__(self, model_paths):
        super().__init__()

        self.models = nn.ModuleList()

        for path in model_paths:

            model = ImprovedHybrid_xLSTM(
                input_size=len(selected_indices),
                embedding_dim=64,
                num_targets=len(PREDICTION_TARGETS)
            ).to(DEVICE)

            model.load_state_dict(
                torch.load(path, map_location=DEVICE)
            )

            model.eval()

            self.models.append(model)

    def forward(self, x, lengths):

        preds = []

        with torch.no_grad():
            for model in self.models:
                preds.append(model(x, lengths))

        preds = torch.stack(preds, dim=0)

        return preds.mean(dim=0)

ensemble_paths = [
    "best_xlstm_seed_42.pth",
    "best_xlstm_seed_123.pth",
    "best_xlstm_seed_456.pth",
    "best_xlstm_seed_789.pth",
    "best_xlstm_seed_1337.pth"
]

ensemble_model = Ensemble_xLSTM(
    ensemble_paths
).to(DEVICE)

print("\n===== ENSEMBLE (5 SEEDS) =====")

ensemble_mae, ensemble_rmse = evaluate_real_metrics(
    ensemble_model,
    test_loader,
    DEVICE
)

ensemble_per_target = evaluate_per_target_metrics(
    ensemble_model,
    test_loader,
    DEVICE,
    PREDICTION_TARGETS
)


===== ENSEMBLE (5 SEEDS) =====
Test MAE  (Real Scale): 1.5591
Test RMSE (Real Scale): 8.6108

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4930
Net_Income   MAE: 1.5899
ROA          MAE: 1.5939


In [ ]:
ensemble_paths = [
    "best_xlstm_seed_42.pth",
    "best_xlstm_seed_123.pth",
    "best_xlstm_seed_789.pth",
    "best_xlstm_seed_1337.pth"
]

ensemble_model = Ensemble_xLSTM(
    ensemble_paths
).to(DEVICE)

print("\n===== ENSEMBLE (4 SEEDS) =====")

ensemble_mae, ensemble_rmse = evaluate_real_metrics(
    ensemble_model,
    test_loader,
    DEVICE
)

ensemble_per_target = evaluate_per_target_metrics(
    ensemble_model,
    test_loader,
    DEVICE,
    PREDICTION_TARGETS
)


===== ENSEMBLE (4 SEEDS) =====
Test MAE  (Real Scale): 1.5508
Test RMSE (Real Scale): 8.6109

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4887
Net_Income   MAE: 1.5808
ROA          MAE: 1.5824


In [ ]:
ensemble_paths = [
    "best_xlstm_seed_42.pth",
    "best_xlstm_seed_789.pth",
    "best_xlstm_seed_1337.pth"
]

ensemble_model = Ensemble_xLSTM(
    ensemble_paths
).to(DEVICE)

print("\n===== ENSEMBLE (3 SEEDS) =====")

ensemble_mae, ensemble_rmse = evaluate_real_metrics(
    ensemble_model,
    test_loader,
    DEVICE
)

ensemble_per_target = evaluate_per_target_metrics(
    ensemble_model,
    test_loader,
    DEVICE,
    PREDICTION_TARGETS
)


===== ENSEMBLE (3 SEEDS) =====
Test MAE  (Real Scale): 1.5514
Test RMSE (Real Scale): 8.6114

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4919
Net_Income   MAE: 1.5813
ROA          MAE: 1.5807


In [ ]:
# =====================================================
# FINAL TRAINING - SEED 1337
# =====================================================

SEED = 1337
seed_everything(SEED)

print(f"Training ImprovedHybrid_xLSTM with seed {SEED}")

# -----------------------------------------------------
# DataLoaders
# -----------------------------------------------------

train_ds = ImprovedYoYDatasetMTL(
    mtl_train_data,
    robust_scaler,
    selected_indices
)

val_ds = ImprovedYoYDatasetMTL(
    mtl_val_data,
    robust_scaler,
    selected_indices
)

test_ds = ImprovedYoYDatasetMTL(
    mtl_test_data,
    robust_scaler,
    selected_indices
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn_improved_mtl
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn_improved_mtl
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn_improved_mtl
)

# -----------------------------------------------------
# Model
# -----------------------------------------------------

model = ImprovedHybrid_xLSTM(
    input_size=len(selected_indices),
    embedding_dim=64,
    num_targets=len(PREDICTION_TARGETS)
).to(DEVICE)

print(
    f"Trainable params: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

# -----------------------------------------------------
# Training
# -----------------------------------------------------

train_info = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=RealScaleL1Loss(),
    lr=0.0044,
    weight_decay=0.0054,
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

print("\nBest epoch:", train_info["best_epoch"])
print("Best validation loss:", train_info["best_val_loss"])

# -----------------------------------------------------
# Load best checkpoint
# -----------------------------------------------------

model.load_state_dict(
    torch.load(
        "best_xlstm_model.pth",
        map_location=DEVICE
    )
)

# -----------------------------------------------------
# Overall metrics
# -----------------------------------------------------

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)

real_mae, real_rmse = evaluate_real_metrics(
    model,
    test_loader,
    DEVICE
)

# -----------------------------------------------------
# Per target metrics
# -----------------------------------------------------

per_target_results = evaluate_per_target_metrics(
    model,
    test_loader,
    DEVICE,
    PREDICTION_TARGETS
)

print("\nPer-target results:")
for target, value in per_target_results.items():
    print(f"{target}: {value:.4f}")

Training ImprovedHybrid_xLSTM with seed 1337
Trainable params: 95,763

Best epoch: 7
Best validation loss: 7.810426544101881

TEST SET RESULTS
Test MAE  (Real Scale): 1.5460
Test RMSE (Real Scale): 8.6086

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4869
Net_Income   MAE: 1.5732
ROA          MAE: 1.5772

Per-target results:
EBITDA: 1.4869
Net_Income: 1.5732
ROA: 1.5772


## Other tries

In [ ]:
class ImprovedHybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int = 64, num_targets: int = len(PREDICTION_TARGETS)):
        super().__init__()
        self.proj = nn.Linear(input_size, embedding_dim)

        # 1) sLSTM config (Strictly Trial 5)
        # Inside the block config we can explicitly override the internal hidden dimension
        # setting it to 32 exactly as Trial 5 suggested, and applying block-level dropout
        slstm_config = sLSTMLayerConfig(
            backend='vanilla',
            hidden_size=32,
            dropout=0.334    # Extracted from Trial 5
        )

        # 2) mLSTM config (Strictly Trial 63)
        # The xLSTM stack requires a shared "embedding_dim" connecting the layers (we use 64).
        # We can reach the desired 128 hidden_size from Trial 63 inside the mLSTM block
        # by using a proj_factor of 2.0 (embedding_dim 64 * 2.0 = 128 inner dimension)
        mlstm_config = mLSTMLayerConfig(
            num_heads=4,
            proj_factor=2.0, # Scales internal capacity to 128
            dropout=0.330    # Extracted from Trial 63
        )

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(
                slstm=slstm_config,
            ),
            mlstm_block=mLSTMBlockConfig(
                mlstm=mlstm_config,
            ),
            context_length=400,
            num_blocks=7,
            embedding_dim=embedding_dim, # 64 -> Transport bus between blocks
            slstm_at=[2, 6]
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.head_dropout = nn.Dropout(p=0.33)
        self.head = nn.Linear(embedding_dim, num_targets)

    def forward(self, x, lengths):
        B, S, _ = x.shape
        out = self.proj(x)
        out = self.xlstm(out)

        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1)

        hidden = self.head_dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

lr_global = 0.0044
weight_decay_global = 0.0054
batch_size_global = 64

print("Training Improved Hybrid xLSTM (Explicitly Split Configurations)")

test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)
train_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_hybrid = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=batch_size_global, shuffle=False, collate_fn=collate_fn_improved_mtl)

hybrid_model = ImprovedHybrid_xLSTM(
    input_size=len(selected_indices),
    embedding_dim=64,
    num_targets=len(PREDICTION_TARGETS)
)

total_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
print(f"Final number of parameters: {total_params:,}")


_ = train_model(
    model=hybrid_model,
    train_loader=train_loader_hybrid,
    val_loader=val_loader_hybrid,
    criterion=RealScaleL1Loss(),
    lr=lr_global,
    weight_decay=weight_decay_global,
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

hybrid_model.load_state_dict(torch.load('best_xlstm_model.pth', map_location=DEVICE))
print("\n--- IMPROVED HYBRID MODEL OVERALL RESULTS ---")
real_mae_hybrid, real_rmse_hybrid = evaluate_real_metrics(hybrid_model, test_loader, DEVICE)
per_target_results_hybrid = evaluate_per_target_metrics(hybrid_model, test_loader, DEVICE, PREDICTION_TARGETS)

Training Improved Hybrid xLSTM (Explicitly Split Configurations)
Final number of parameters: 219,755

--- IMPROVED HYBRID MODEL OVERALL RESULTS ---
Test MAE  (Real Scale): 1.5534
Test RMSE (Real Scale): 8.6116

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.4972
Net_Income   MAE: 1.5869
ROA          MAE: 1.5756


# Rolling validation - ImprovedHybrid_xLSTM

Temporal rolling validation with folds 2018/2019, 2019/2020 and 2020/2021. Each fold refits the scaler only on its training years and trains the Best model architecture.


In [ ]:
# Rolling validation for the Best model architecture.
# This redefines ImprovedHybrid_xLSTM to match the first code cell after "# Best model".
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class ImprovedHybrid_xLSTM(nn.Module):
    def __init__(self, input_size: int, embedding_dim: int = 64, num_targets: int = len(PREDICTION_TARGETS)):
        super().__init__()
        self.proj = nn.Linear(input_size, embedding_dim)

        slstm_config = sLSTMLayerConfig(
            backend='vanilla',
            hidden_size=32,
            dropout=0.334
        )

        mlstm_config = mLSTMLayerConfig(
            num_heads=4,
            proj_factor=2.0,
            dropout=0.330
        )

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mlstm_config),
            context_length=400,
            num_blocks=3,
            embedding_dim=embedding_dim,
            slstm_at=[2]
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.head_dropout = nn.Dropout(p=0.33)
        self.head = nn.Linear(embedding_dim, num_targets)

    def forward(self, x, lengths):
        out = self.proj(x)
        out = self.xlstm(out)
        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1)
        hidden = self.head_dropout(last_out)
        logits = self.head(hidden)
        if logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        return logits

ROLLING_FOLDS = [
    {"fold": "fold_1", "train_until": 2018, "val_year": 2019},
    {"fold": "fold_2", "train_until": 2019, "val_year": 2020},
    {"fold": "fold_3", "train_until": 2020, "val_year": 2021},
]

ROLLING_SEEDS = [42, 123, 456, 789, 1337]
ROLLING_EPOCHS = MAX_EPOCHS
ROLLING_PATIENCE = PATIENCE
ROLLING_TEST_AFTER_YEAR = 2021

lr_global = 0.0044
weight_decay_global = 0.0054
batch_size_global = 64


def make_fold_loaders(data_df, train_until, val_year, batch_size):
    fold_train = data_df[data_df["year"] <= train_until].copy()
    fold_val = data_df[data_df["year"] == val_year].copy()
    fold_test = data_df[data_df["year"] > ROLLING_TEST_AFTER_YEAR].copy()

    if fold_train.empty or fold_val.empty:
        raise ValueError(f"Empty rolling fold: train_until={train_until}, val_year={val_year}")

    fold_scaler = RobustScaler()
    fold_train_windows = np.vstack([row[:, selected_indices] for row in fold_train["window_data"]])
    fold_scaler.fit(fold_train_windows)

    train_mtl = extract_mtl_df(fold_train)
    val_mtl = extract_mtl_df(fold_val)
    test_mtl = extract_mtl_df(fold_test) if not fold_test.empty else pd.DataFrame()

    train_loader = DataLoader(
        ImprovedYoYDatasetMTL(train_mtl, fold_scaler, selected_indices),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn_improved_mtl,
    )
    val_loader = DataLoader(
        ImprovedYoYDatasetMTL(val_mtl, fold_scaler, selected_indices),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn_improved_mtl,
    )
    test_loader = None
    if not test_mtl.empty:
        test_loader = DataLoader(
            ImprovedYoYDatasetMTL(test_mtl, fold_scaler, selected_indices),
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn_improved_mtl,
        )

    return train_loader, val_loader, test_loader, len(train_mtl), len(val_mtl), len(test_mtl)


def train_improved_hybrid_in_memory(train_loader, val_loader, seed):
    seed_everything(seed)
    model = ImprovedHybrid_xLSTM(
        input_size=len(selected_indices),
        embedding_dim=64,
        num_targets=len(PREDICTION_TARGETS),
    ).to(DEVICE)

    criterion = RealScaleL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_global, weight_decay=weight_decay_global)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_loss = float('inf')
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(ROLLING_EPOCHS):
        model.train()
        for x_batch, y_batch, mask_batch, lengths in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            mask_batch = mask_batch.to(DEVICE)
            lengths = lengths.to(DEVICE)

            if mask_batch.sum() == 0:
                continue

            optimizer.zero_grad(set_to_none=True)
            preds = model(x_batch, lengths)
            loss = criterion(preds[mask_batch], y_batch[mask_batch])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        val_valid_count = 0
        with torch.no_grad():
            for x_batch, y_batch, mask_batch, lengths in val_loader:
                x_batch = x_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                mask_batch = mask_batch.to(DEVICE)
                lengths = lengths.to(DEVICE)

                if mask_batch.sum() == 0:
                    continue

                preds = model(x_batch, lengths)
                valid_count = mask_batch.sum().item()
                loss = criterion(preds[mask_batch], y_batch[mask_batch])
                val_loss += loss.item() * valid_count
                val_valid_count += valid_count

        val_loss /= max(1, val_valid_count)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= ROLLING_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_loss


def evaluate_real_metrics_table(model, loader, device):
    if loader is None:
        return float('nan'), float('nan')

    model.eval()
    total_absolute_error = 0.0
    total_squared_error = 0.0
    total_values = 0

    with torch.no_grad():
        for x_batch, y_batch_symlog, mask_batch, lengths in loader:
            x_batch = x_batch.to(device)
            y_batch_symlog = y_batch_symlog.to(device)
            mask_batch = mask_batch.to(device)
            lengths = lengths.to(device)

            if mask_batch.sum() == 0:
                continue

            preds_symlog = model(x_batch, lengths)
            valid_preds_symlog = preds_symlog[mask_batch]
            valid_y_symlog = y_batch_symlog[mask_batch]

            preds_real = inverse_symlog_tensor(valid_preds_symlog)
            y_real = inverse_symlog_tensor(valid_y_symlog)
            errors = preds_real - y_real

            total_absolute_error += torch.abs(errors).sum().item()
            total_squared_error += torch.pow(errors, 2).sum().item()
            total_values += valid_y_symlog.numel()

    mae = total_absolute_error / max(1, total_values)
    rmse = (total_squared_error / max(1, total_values)) ** 0.5
    return mae, rmse


rolling_results = []

for fold_cfg in ROLLING_FOLDS:
    train_loader, val_loader, test_loader, n_train, n_val, n_test = make_fold_loaders(
        improved_data_df,
        train_until=fold_cfg["train_until"],
        val_year=fold_cfg["val_year"],
        batch_size=batch_size_global,
    )

    print(f"\n========== {fold_cfg['fold']} | train <= {fold_cfg['train_until']} | val = {fold_cfg['val_year']} ==========")
    print(f"MTL samples -> train: {n_train}, val: {n_val}, final test > {ROLLING_TEST_AFTER_YEAR}: {n_test}")

    for seed in ROLLING_SEEDS:
        model, best_val_loss = train_improved_hybrid_in_memory(train_loader, val_loader, seed)
        val_mae, val_rmse = evaluate_real_metrics_table(model, val_loader, DEVICE)
        test_mae, test_rmse = evaluate_real_metrics_table(model, test_loader, DEVICE)

        rolling_results.append({
            "fold": fold_cfg["fold"],
            "train_until": fold_cfg["train_until"],
            "val_year": fold_cfg["val_year"],
            "seed": seed,
            "best_val_loss": best_val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "test_after_2021_mae": test_mae,
            "test_after_2021_rmse": test_rmse,
        })

        print(
            f"Seed {seed} -> val MAE={val_mae:.4f}, val RMSE={val_rmse:.4f}, "
            f"test>2021 MAE={test_mae:.4f}, test>2021 RMSE={test_rmse:.4f}"
        )

rolling_results_df = pd.DataFrame(rolling_results)
display(rolling_results_df)

rolling_summary_df = (
    rolling_results_df
    .groupby(["fold", "train_until", "val_year"], as_index=False)
    .agg(
        val_mae_mean=("val_mae", "mean"),
        val_mae_std=("val_mae", "std"),
        val_rmse_mean=("val_rmse", "mean"),
        test_after_2021_mae_mean=("test_after_2021_mae", "mean"),
        test_after_2021_mae_std=("test_after_2021_mae", "std"),
    )
)

display(rolling_summary_df)




========== fold_1 | train <= 2018 | val = 2019 ==========
MTL samples -> train: 922, val: 141, final test > 2021: 485
Seed 42 -> val MAE=1.7963, val RMSE=8.2043, test>2021 MAE=1.5597, test>2021 RMSE=8.6096
Seed 123 -> val MAE=1.8093, val RMSE=8.2014, test>2021 MAE=1.5591, test>2021 RMSE=8.6096
Seed 456 -> val MAE=1.8060, val RMSE=8.2072, test>2021 MAE=1.6009, test>2021 RMSE=8.6250
Seed 789 -> val MAE=1.8042, val RMSE=8.2014, test>2021 MAE=1.5724, test>2021 RMSE=8.6151
Seed 1337 -> val MAE=1.8105, val RMSE=8.2055, test>2021 MAE=1.5630, test>2021 RMSE=8.6140

========== fold_2 | train <= 2019 | val = 2020 ==========
MTL samples -> train: 1063, val: 147, final test > 2021: 485
Seed 42 -> val MAE=9.3982, val RMSE=105.9259, test>2021 MAE=1.5604, test>2021 RMSE=8.6154
Seed 123 -> val MAE=9.3772, val RMSE=105.9363, test>2021 MAE=1.5678, test>2021 RMSE=8.6159
Seed 456 -> val MAE=9.3751, val RMSE=105.9397, test>2021 MAE=1.5860, test>2021 RMSE=8.6187
Seed 789 -> val MAE=9.3735, val RMSE=105.941

,fold,train_until,val_year,seed,best_val_loss,val_mae,val_rmse,test_after_2021_mae,test_after_2021_rmse
0,fold_1,2018,2019,42,1.796347,1.796347,8.204340,1.559742,8.609594
1,fold_1,2018,2019,123,1.809323,1.809323,8.201441,1.559103,8.609562
2,fold_1,2018,2019,456,1.806025,1.806025,8.207215,1.600946,8.625020
3,fold_1,2018,2019,789,1.804192,1.804192,8.201363,1.572373,8.615142
4,fold_1,2018,2019,1337,1.810482,1.810482,8.205508,1.562982,8.614004
5,fold_2,2019,2020,42,9.398157,9.398157,105.925907,1.560432,8.615424
6,fold_2,2019,2020,123,9.377243,9.377244,105.936283,1.567788,8.615941
7,fold_2,2019,2020,456,9.375103,9.375103,105.939697,1.585956,8.618659
8,fold_2,2019,2020,789,9.373514,9.373515,105.941444,1.590221,8.626665
9,fold_2,2019,2020,1337,9.372181,9.372180,105.941293,1.565065,8.616017


,fold,train_until,val_year,val_mae_mean,val_mae_std,val_rmse_mean,test_after_2021_mae_mean,test_after_2021_mae_std
0,fold_1,2018,2019,1.805274,0.005589,8.203974,1.571029,0.017544
1,fold_2,2019,2020,9.379240,0.010742,105.936925,1.573893,0.013309
2,fold_3,2020,2021,6.300816,0.022797,63.072043,1.576488,0.042133
